# FE-1 — Face Embedding

Extract a 512-dimensional face embedding from an image.

**Model:** `FE-1`  
**Input:** Image files (`.jpg`, `.jpeg`, `.png`, etc.)

---
### Contents
1. Setup
2. One-call embedding extraction
3. Two-step: upload now, poll later
4. Error handling
5. Async client

---
## 1. Setup

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('../src'))

from authenta.authenta_client import AuthentaClient
from authenta import (
    AuthentaError,
    AuthenticationError,
    QuotaExceededError,
    InsufficientCreditsError,
)

CLIENT_ID     = "YOUR_CLIENT_ID"
CLIENT_SECRET = "YOUR_CLIENT_SECRET"
BASE_URL      = "https://platform.authenta.ai"

client = AuthentaClient(
    base_url=BASE_URL,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)

print("Client ready.")

---
## 2. One-call Embedding Extraction

`client.extract_face_vector()` uploads the image, waits for processing, and returns embedding.

In [ ]:
IMAGE = "../data_samples/nano_img.png"

media = client.extract_face_vector(
    img_path=IMAGE,
    auto_polling=True
)

embedding = media["result"]["embedding"]

print(f"Media ID   : {media['mid']}")
print(f"Status     : {media['status']}")
print(f"Vector Dim : {len(embedding)}")

---
## 3. Two-step: Upload Now, Poll Later

Use `auto_polling=False` to return immediately, then fetch result manually.

In [ ]:
# Step 1 — upload
upload_meta = client.extract_face_vector(
    img_path=IMAGE,
    auto_polling=False
)

mid = upload_meta["mid"]
print(f"Uploaded. Media ID : {mid}")

In [ ]:
# Step 2 — poll
media = client.wait_for_media(mid)

# Step 3 — fetch result
result = client.get_result(media)
embedding = result["embedding"]

print(f"Status     : {media['status']}")
print(f"Vector Dim : {len(embedding)}")

---
## 4. Error Handling

In [ ]:
try:
    media = client.extract_face_vector(IMAGE)
    print(len(media["result"]["embedding"]))

except AuthenticationError:
    print("Authentication failed — check credentials.")
except QuotaExceededError:
    print("Quota exceeded.")
except InsufficientCreditsError:
    print("Not enough credits.")
except TimeoutError as e:
    print(f"Timed out: {e}")
except AuthentaError as e:
    print(f"API error [{e.code}]: {e.message}")

---
## 5. Async Client

In [ ]:
import asyncio
from authenta.async_authenta_client import AsyncAuthentaClient

async def extract_embedding():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:

        media = await client.extract_face_vector(
            img_path=IMAGE,
            auto_polling=True
        )

        embedding = media["result"]["embedding"]

        print(f"Status     : {media['status']}")
        print(f"Vector Dim : {len(embedding)}")

await extract_embedding()